### Transformations applied to Sprints Dataset 


The notebook below accomplishes the following:


1. Read bronze `sprints` table
1. Kept columns required for analytics (Dropped `url` column)
1. Standardized column names using snake_case 
1. Renamed columns to make them more meaningful 
1. Filtered out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Removed duplicate records
1. Transformed values of column `race_name` to Title Case
1. Wrote the transformed data into silver layer

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
# Read & Write tables for Races

bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"

In [0]:
sprints_df=(
    spark.read.table(bronze_table)
                .drop("url")
                .withColumnsRenamed(
                        {'date':'race_date',
                        'raceName':'race_name',
                        'constructorId':'constructor_Id',
                        'driverId':'driver_Id',
                        'grid':'grid_position',
                        'laps':'completed_laps',
                        'number':'car_number',
                        'poisition':'final_position',
                        'positionText':'final_position_text'
                        }

))
display(sprints_df)

In [0]:
# finding Duplicates

#'''
dupes= (sprints_df.groupBy(sprints_df.columns)
.count()
.filter(F.col("count")>1))
display(dupes)
#'''


In [0]:
# Chaining filters and dropping duplicates

sprints_valid_df= (
    sprints_df
            .filter(F.col('season').isNotNull() &
                    F.col('round').isNotNull() &
                    F.col('constructor_id').isNotNull() &
                    F.col('driver_id').isNotNull())
            .dropDuplicates(['season','round','constructor_id','driver_id'])
            #.dropDuplicates()
)

In [0]:
#Count of duplicate records

display(sprints_df.count()-sprints_valid_df.count())

In [0]:
#Standardizing Column naming convention
sprints_final_df=(sprints_valid_df
                                .withColumn("race_name",F.initcap(F.col("race_name")))    
                )

In [0]:
#Writing Final table into Silver Layer

(

sprints_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(silver_table)
)


In [0]:
display(sprints_final_df)